In [1]:
#format the book
%matplotlib inline
import sys
sys.path.insert(0, '..')
import book_format
book_format.set_style()

# 互動範例 Interactions

這是一組互動範例（interaction），大多來自本書。如果你讀的是本書的印刷版，或是透過 Github 或 nbviewer 線上閱讀，你將無法執行這些互動範例。

所以我建立了這個 notebook。如果你的電腦沒有安裝 IPython，以下是執行互動範例的方法。

1. 在瀏覽器中前往 try.juptyer.org。它會為你啟動一個暫時的 notebook 伺服器。

2. 點選 **New** 按鈕並選擇 `Python 3`。這會建立一個新的 notebook，在你的瀏覽器中為你執行 Python 3。

3. 複製這個 notebook 中某個 cell 的全部內容，貼到瀏覽器上那個 notebook 的 'code' cell 裡。 

4. 按 CTRL+ENTER 執行該 cell。

5. 玩得開心！改改程式碼。玩一玩。做實驗。動手 hack。

你的伺服器和 notebook 不會被永久保存。一旦關閉工作階段，你的資料就會遺失。沒錯，按下儲存時它會說正在儲存你的檔案，你也能在目錄中看到它。但那只是發生在一個 Docker 容器裡，你一關閉視窗它就會被刪除。任何想保留的修改，請複製貼上到外部檔案。

當然，如果你已安裝 IPython，就可以下載這個 notebook 並在自己的電腦上執行。在下載此檔案的目錄中，於命令提示字元輸入

    ipython notebook
    
然後點選這個檔案的名稱來開啟它。

# 實驗 FPF' Experimenting with FPF'


Kalman filter（卡爾曼濾波器）在 prediction step（預測步驟）中使用方程式 $P^- = FPF^\mathsf{T}$ 來計算 covariance matrix（共變異數矩陣）的 prior（先驗），其中 P 是共變異數矩陣，F 是系統的 state transition function（狀態轉移函數）。對於牛頓系統 $x = \dot{x}\Delta t + x_0$，F 可能長這樣

$$F = \begin{bmatrix}1 & \Delta t\\0 & 1\end{bmatrix}$$

$FPF^\mathsf{T}$ 藉由納入位置（$x$）與速度（$\dot{x}$）之間的 correlation（相關性）來改變 P。這個互動圖讓你看到不同的 F 設計對這個值的影響。例如，

* 如果 $x$ 與 $\dot{x}$ 不相關呢？（將 F01 設為 0）

* 如果 $x = 2\dot{x}\Delta t + x_0$ 呢？（將 F01 設為 2）

*  如果 $x = \dot{x}\Delta t + 2*x_0$ 呢？（將 F00 設為 2）

* 如果 $x = \dot{x}\Delta t$ 呢？ （將 F00 設為 0）

In [2]:
%matplotlib inline
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
import numpy as np
import numpy.linalg as linalg
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(x, P, edgecolor='k'):
    U,s,v = linalg.svd(P)
    angle = math.atan2(U[1,0],U[0,0])
    width  = math.sqrt(s[0]) * 2
    height = math.sqrt(s[1]) * 2

    ax = plt.gca()
    e = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                edgecolor=edgecolor, facecolor='none',
                lw=2, ls='solid')
    ax.add_patch(e)
    ax.set_aspect('equal')
    
    
def plot_FPFT(F00, F01, F10, F11, covar):
    
    dt = 1.
    x = np.array((0, 0.))
    P = np.array(((1, covar), (covar, 2)))
    F = np.array(((F00, F01), (F10, F11)))

    plot_covariance_ellipse(x, P)
    plot_covariance_ellipse(x, np.dot(F, P).dot(F.T), edgecolor='r')
    #plt.axis('equal')
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)
    plt.title(str(F))
    plt.xlabel('position')
    plt.ylabel('velocity')
    plt.show()
                 
interact(plot_FPFT, 
         F00=widgets.IntSlider(value=1, min=0, max=2.), 
         F01=widgets.FloatSlider(value=1, min=0., max=2., description='F01(dt)'),
         F10=widgets.FloatSlider(value=0, min=0., max=2.),
         F11=widgets.FloatSlider(value=1, min=0., max=2.),
         covar=widgets.FloatSlider(value=0, min=0, max=1.));

interactive(children=(IntSlider(value=1, description='F00', max=2), FloatSlider(value=1.0, description='F01(dt…

# Covariance Ellipse（共變異數橢圓）

觀察改變下列形式之共變異數矩陣中的 variance（變異數）與 covariance（共變異數）所產生的效果

$$\begin{bmatrix}\texttt{var}_x & \texttt{cov}_xy \\ \texttt{cov}_xy & \texttt{var}_y\end{bmatrix}$$

In [3]:
%matplotlib inline
from ipywidgets import interact, interactive, fixed
from ipywidgets import FloatSlider
from math import cos, sin, pi, atan2, sqrt
import  numpy.linalg as linalg
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(P):
    U,s,v = linalg.svd(P)
    angle = atan2(U[1,0],U[0,0])
    width  = sqrt(s[0]) * 2
    height = sqrt(s[1]) * 2

    ax = plt.gca()
    e = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                edgecolor='k', facecolor='none',
                lw=2, ls='solid')
    ax.add_patch(e)
    h, w = height/4, width/4
    plt.plot([0, h*cos(angle+pi/2)], [0, h*sin(angle+pi/2)])
    plt.plot([0, w*cos(angle)],      [0, w*sin(angle)])

def plot_covariance(var_x, var_y, cov_xy):
    P = [[var_x, cov_xy], [cov_xy, var_y]]
    plot_covariance_ellipse(P)
    plt.xlim(-6, 6)
    plt.gca().set_aspect('equal')
    plt.ylim(-6, 6)
    plt.show()

interact (plot_covariance,           
          var_x=FloatSlider(value=5., min=0, max=20.), 
          var_y=FloatSlider(value=5., min=0., max=20.), 
          cov_xy=FloatSlider(value=1.5, min=0.0, max=50, step=.2));


interactive(children=(FloatSlider(value=5.0, description='var_x', max=20.0), FloatSlider(value=5.0, descriptio…

# g-h Filter（g-h 濾波器）

用各種不同的值來實驗 g-h filter 的參數。

In [4]:
%matplotlib inline
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import numpy.random as random

def gen_data(x0, dx, count, noise_factor):
    return [x0 + dx*i + random.randn()*noise_factor for i in range (count)]

def g_h_filter(data, x0, dx, g, h, dt=1., pred=None):    
    x = x0
    results = []
    for z in data:
        #prediction step
        x_est = x + (dx*dt)
        dx = dx        
        if pred is not None:
            pred.append(x_est)
        
        # update step
        residual = z - x_est
        dx = dx    + h * (residual) / dt
        x  = x_est + g * residual     
        results.append(x)  
    return np.array(results)

zs = gen_data(x0=5, dx=5, count=100, noise_factor=50)

def interactive_gh(x, dx, g, h):
    data = g_h_filter(data=zs, x0=x, dx=dx, dt=1.,g=g, h=h)
    plt.plot(zs, color='r')
    plt.plot(data, color='k')
    plt.show()

interact (interactive_gh,           
          x=widgets.FloatSlider(value=0., min=-50, max=50.), 
          dx=widgets.FloatSlider(value=5., min=-50., max=50.), 
          g=widgets.FloatSlider(value=0.1, min=0.01, max=2, step=.02), 
          h=widgets.FloatSlider(value=0.02, min=0.0, max=0.5, step=0.01));

interactive(children=(FloatSlider(value=0.0, description='x', max=50.0, min=-50.0), FloatSlider(value=5.0, des…